In [7]:
import requests
from bs4 import BeautifulSoup
import json
import os

pwd = os.path.dirname(os.getcwd())

searchable_pages = []

for dir in ['control','dynsys']:
    dir_path = os.path.join(pwd, "src/pages", dir)
    for page in os.listdir(dir_path):
        searchable_pages.append(os.path.join(dir, page.replace(".mdx", '')))

page_text = []

for page in searchable_pages:
    print(page)
    url = f"http://localhost:4324/{page}"
    html = requests.get(url).text


    soup = BeautifulSoup(html, features="html.parser")

    # kill all script and style elements
    for script in soup(["script", "style", "nav", 'header', 'footer']):
        script.extract()    # rip it out

    try:
        soup.find("div", attrs={"id": "nav_container"}).extract()
    except:
        pass

    try:
        soup.find("h1", attrs={"id": "page_title"}).extract()
    except:
        pass

    try:
        soup.find("div", attrs={"id": "title_bar"}).extract()
    except:
        pass

    # get text
    text = soup.get_text()

    # break into lines and remove leading and trailing space on each
    lines = (line.strip() for line in text.splitlines())
    # break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # drop blank lines
    text = ' '.join(chunk for chunk in chunks if chunk)

    banned_strings = [
        "Derivation +",
        "Solution +",
        "All courses Statics Dynamics Solid Mechanics ",
        "Scroll back to top"
    ]

    for b in banned_strings:    
        text = text.replace(b, "")

    page_text.append({"title": page.split("/")[1].replace("_", " ").capitalize(),"text": text, 'link': "/" + page, 'course': page.split('/')[0]})

json.dump(page_text, open("../src/search.json", "w"), indent=2)

control/tracking
control/optimal-controllers
control/matrix-exponential
control/state-space-models
control/images
control/optimal-observers
control/state-estimation
control/pid
control/eigenvalue-placement
control/stability
dynsys/accelerated-frames
dynsys/app-rigid-body
dynsys/app-vibration
dynsys/images
dynsys/hamilton
dynsys/app-robotics
dynsys/app-flight-mechanics
dynsys/kinematics
dynsys/newton
dynsys/dynamical-systems
dynsys/lagrange
